# Algoritmos Genéticos con Fitness Sharing (Niching)
**Curso:** Computación Bioinspirada  
**Tema:** Algoritmos Genéticos y Modelos de Computación Evolutiva  
**Técnica central:** Fitness Sharing para identificación de múltiples óptimos

---

## ¿Qué problema resolvemos?

La optimización clásica busca **un único óptimo global**. Pero en muchos problemas reales existen **múltiples soluciones igualmente válidas** (óptimos locales de alta calidad). Un AG estándar tiende a **converger prematuramente** a uno solo de esos picos, descartando el resto.

**Ejemplo concreto:** en diseño de fármacos, una molécula puede tener varias conformaciones estructurales que interactúan bien con un receptor. Nos interesa conocer todas, no solo la mejor.

### Solución: técnicas de *Niching*
El *niching* permite que la población se **subdivida en subpoblaciones** que habitan diferentes regiones del espacio de búsqueda. Cada subpoblación converge a un pico distinto.

La técnica usada aquí es **Fitness Sharing**: reducir el fitness de un individuo si hay muchos individuos similares cerca de él, penalizando la sobrepoblación en un mismo pico.

---
## Sección 0 — Imports y configuración global

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# Semilla para reproducibilidad (puedes cambiarla o eliminarla)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Librerías cargadas correctamente.")

---
## Sección 1 — Función objetivo (código original de referencia)

### ¿Qué hace?
Define la **función multimodal original** del enunciado, con 3 picos gaussianos en x=2, x=5 y x=8.

### ¿Por qué gaussianas?
Una gaussiana `A · exp(-((x - μ)² / (2σ²)))` crea un pico suave y acotado:
- `μ` controla la **posición** del pico
- `σ` controla el **ancho** del pico
- `A` controla la **altura** (amplitud)

Sumando varias gaussianas obtenemos una función multimodal continua y diferenciable, ideal para probar niching.

In [ ]:
def objective_function_original(x):
    """Función objetivo original del enunciado (3 picos)."""
    x = np.array(x)
    term1 = 1.0 * np.exp(-((x - 2)**2) / (2 * 0.3**2))   # Pico alto en x=2
    term2 = 0.8 * np.exp(-((x - 5)**2) / (2 * 0.5**2))   # Pico medio en x=5
    term3 = 0.9 * np.exp(-((x - 8)**2) / (2 * 0.4**2))   # Pico alto-medio en x=8
    return term1 + term2 + term3

# Visualización de la función original
x_plot = np.linspace(0, 10, 500)
y_plot = objective_function_original(x_plot)

plt.figure(figsize=(10, 4))
plt.plot(x_plot, y_plot, 'b-', linewidth=2)
plt.title("Función Objetivo Original (referencia del enunciado)")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid(True)
for peak_x in [2, 5, 8]:
    plt.axvline(peak_x, color='gray', linestyle='--', alpha=0.5)
    plt.text(peak_x + 0.05, 0.05, f'x={peak_x}', color='gray')
plt.tight_layout()
plt.show()

---
## Sección 2 — Nueva función objetivo (≥ 4 picos, dos cercanos entre sí)

### ¿Qué se modifica y por qué?
El enunciado pide una función **más compleja** con estas características:

1. **Al menos 4 picos** — para que el algoritmo tenga más óptimos que encontrar
2. **Al menos dos picos cercanos** — la distancia entre ellos debe ser **menor que el SIGMA_SHARE original (0.75)**, para poder estudiar el efecto del parámetro cuando es "grande" vs "pequeño" respecto a esa distancia

### Diseño de la nueva función
Se definen **4 picos gaussianos**:

| Pico | Posición (μ) | Altura (A) | Ancho (σ) | Nota |
|------|-------------|-----------|-----------|------|
| 1    | x = 1.5     | 1.0       | 0.25      | Pico estrecho y alto |
| 2    | x = 4.0     | 0.85      | 0.35      | Pico cercano al siguiente |
| 3    | x = 4.7     | 0.80      | 0.30      | **Pico cercano** (distancia ≈ 0.7 al anterior) |
| 4    | x = 7.5     | 0.95      | 0.40      | Pico ancho y alto |
| 5    | x = 9.5     | 0.60      | 0.20      | Pico pequeño y estrecho |

> Los picos en x=4.0 y x=4.7 están separados por **~0.7 unidades**, que es menor que el SIGMA_SHARE original (0.75). Esto crea la situación de picos cercanos que el enunciado requiere para el experimento.

In [ ]:
def objective_function_nueva(x):
    """
    Nueva función objetivo con 5 picos gaussianos.
    
    Picos:
      - x=1.5  (alto, estrecho)
      - x=4.0  (medio-alto)
      - x=4.7  (medio, CERCANO al anterior — distancia ~0.7 < SIGMA_SHARE original 0.75)
      - x=7.5  (alto, ancho)
      - x=9.5  (bajo, estrecho)
    """
    x = np.array(x)
    pico1 = 1.00 * np.exp(-((x - 1.5)**2) / (2 * 0.25**2))
    pico2 = 0.85 * np.exp(-((x - 4.0)**2) / (2 * 0.35**2))
    pico3 = 0.80 * np.exp(-((x - 4.7)**2) / (2 * 0.30**2))  # ← pico cercano a pico2
    pico4 = 0.95 * np.exp(-((x - 7.5)**2) / (2 * 0.40**2))
    pico5 = 0.60 * np.exp(-((x - 9.5)**2) / (2 * 0.20**2))
    return pico1 + pico2 + pico3 + pico4 + pico5


# Definición de los picos (para referencia en gráficos)
PICOS_REALES = {
    'x': [1.5, 4.0, 4.7, 7.5, 9.5],
    'labels': ['P1\n(x=1.5)', 'P2\n(x=4.0)', 'P3\n(x=4.7)', 'P4\n(x=7.5)', 'P5\n(x=9.5)']
}

# Visualización de la nueva función
x_plot = np.linspace(0, 10, 1000)
y_plot = objective_function_nueva(x_plot)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x_plot, y_plot, 'b-', linewidth=2.5, label='f(x) nueva')

# Marcar picos reales
for px, lbl in zip(PICOS_REALES['x'], PICOS_REALES['labels']):
    py = objective_function_nueva(px)
    ax.plot(px, py, 'g*', markersize=14)
    ax.axvline(px, color='green', linestyle='--', alpha=0.3)
    ax.text(px, py + 0.03, lbl, ha='center', fontsize=8, color='darkgreen')

# Resaltar la zona de picos cercanos
ax.axvspan(3.5, 5.2, alpha=0.08, color='orange', label='Zona picos cercanos (P2–P3)')
ax.annotate('Distancia ≈ 0.7\n(< SIGMA_SHARE original 0.75)',
            xy=(4.35, 0.55), fontsize=9, color='darkorange',
            ha='center', style='italic')

ax.set_title("Nueva Función Objetivo — 5 picos (dos cercanos entre sí)", fontsize=13)
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

print("Valores en los picos reales:")
for px, lbl in zip(PICOS_REALES['x'], PICOS_REALES['labels']):
    print(f"  {lbl.replace(chr(10),' ')}: f({px}) = {objective_function_nueva(px):.4f}")

### Análisis de la función

La función tiene **5 picos** diferenciables, con las siguientes propiedades relevantes para el experimento:

- **Picos P2 (x=4.0) y P3 (x=4.7)** están separados por solo **~0.7 unidades**. Esto es menor que el `SIGMA_SHARE` original (0.75), lo que los convierte en candidatos a **fusionarse en un solo nicho** si el radio es demasiado grande.
- El resto de picos están separados por más de 2 unidades entre sí, suficiente para que casi cualquier valor razonable de SIGMA_SHARE los mantenga separados.
- Existe un pico menor (P5, x=9.5) que evalúa si el algoritmo da representación a soluciones de menor altura.

---
## Sección 3 — Funciones del Algoritmo Genético

Estas funciones son el **núcleo del AG**. Se mantienen igual al código base del enunciado, pero con documentación extendida para entender qué hace cada una.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.1 Inicialización de la población
# ─────────────────────────────────────────────────────────────────────────────
def initialize_population(size, lower_bound, upper_bound):
    """
    Crea una población inicial de individuos distribuidos aleatoriamente
    de forma uniforme en el intervalo [lower_bound, upper_bound].
    
    En este problema 1D, cada individuo es un único número real (valor de x).
    Representa una posible solución candidata en el espacio de búsqueda.
    
    La distribución uniforme garantiza cobertura inicial de todo el dominio,
    lo que es importante para que el AG pueda 'detectar' todos los picos.
    """
    return [random.uniform(lower_bound, upper_bound) for _ in range(size)]


# ─────────────────────────────────────────────────────────────────────────────
# 3.2 Cálculo de fitness original
# ─────────────────────────────────────────────────────────────────────────────
def calculate_fitness(population, objective_fn):
    """
    Evalúa la función objetivo para cada individuo de la población.
    
    El 'fitness' es la medida de qué tan buena es una solución.
    Aquí fitness = f(x): cuanto más alto el pico en que cae el individuo,
    mayor su fitness.
    
    Este es el fitness REAL, antes de aplicar Fitness Sharing.
    """
    return [objective_fn(ind) for ind in population]


# ─────────────────────────────────────────────────────────────────────────────
# 3.3 Distancia fenotípica
# ─────────────────────────────────────────────────────────────────────────────
def phenotypic_distance(ind1, ind2):
    """
    Calcula la distancia entre dos individuos en el espacio del fenotipo.
    
    Para este problema 1D, el fenotipo ES el valor de x directamente,
    por lo que la distancia es simplemente |x1 - x2|.
    
    En problemas multidimensionales se usaría distancia euclidiana.
    En problemas con cromosoma binario se podría usar distancia de Hamming.
    """
    return abs(ind1 - ind2)


# ─────────────────────────────────────────────────────────────────────────────
# 3.4 Fitness Sharing — el corazón del niching
# ─────────────────────────────────────────────────────────────────────────────
def apply_fitness_sharing(population, fitness_values, sigma_share, alpha_sharing=1.0):
    """
    Ajusta el fitness de cada individuo penalizando la 'sobrepoblación' en su nicho.
    
    Mecánica:
    ---------
    Para cada individuo i:
      1. Se calcula su 'niche_count': la suma de contribuciones sh(d_ij) 
         de todos los vecinos j dentro del radio sigma_share.
      
         sh(d) = 1 - (d / sigma_share)^alpha   si d < sigma_share
         sh(d) = 0                              si d >= sigma_share
         
         Nota: sh(0) = 1 (uno mismo contribuye con 1, siempre)
               sh(sigma_share - ε) → 0 (un vecino en el borde contribuye ~0)
      
      2. El fitness compartido = fitness_original / niche_count
    
    Efecto:
    -------
    Si hay 10 individuos en el mismo pico, cada uno tiene niche_count ≈ 10,
    y su fitness compartido es ~1/10 del original.
    Un individuo en un pico solitario tiene niche_count ≈ 1, 
    y su fitness compartido ≈ su fitness original.
    
    Esto hace que el AG prefiera mantener representantes en TODOS los picos,
    no solo en el más alto.
    
    Parámetros:
    -----------
    sigma_share  : radio del nicho (determina qué tan lejos 'se ven' dos individuos)
    alpha_sharing: forma de la función de compartición (1.0 = lineal)
    """
    shared_fitness_values = []
    num_individuals = len(population)
    
    for i in range(num_individuals):
        niche_count = 0
        for j in range(num_individuals):
            dist = phenotypic_distance(population[i], population[j])
            if dist < sigma_share:
                # Contribución del vecino j al nicho de i
                sh_d = 1 - (dist / sigma_share) ** alpha_sharing
                niche_count += sh_d
            # Si dist >= sigma_share, sh_d = 0 → no contribuye
        
        if niche_count > 1e-6:
            shared_fitness_values.append(fitness_values[i] / niche_count)
        else:
            shared_fitness_values.append(fitness_values[i])  # caso degenerad
    
    return shared_fitness_values


# ─────────────────────────────────────────────────────────────────────────────
# 3.5 Selección por torneo
# ─────────────────────────────────────────────────────────────────────────────
def tournament_selection(population, fitness_values, k):
    """
    Selecciona un padre mediante torneo de tamaño k.
    
    Mecanismo:
    - Se eligen k individuos al azar de la población.
    - El individuo con mayor fitness (COMPARTIDO) entre ellos gana.
    
    Usar el fitness COMPARTIDO (no el original) es clave:
    asegura que individuos en picos menos poblados tengan chances reales
    de convertirse en padres, promoviendo la diversidad.
    
    Mayor k = mayor presión selectiva (el mejor casi siempre gana).
    k=2 o k=3 es típico para un balance presión/diversidad.
    """
    selected_indices = random.sample(range(len(population)), k)
    tournament_fitness = [fitness_values[i] for i in selected_indices]
    winner_idx = np.argmax(tournament_fitness)
    return population[selected_indices[winner_idx]]


# ─────────────────────────────────────────────────────────────────────────────
# 3.6 Crossover aritmético
# ─────────────────────────────────────────────────────────────────────────────
def crossover(parent1, parent2):
    """
    Crossover aritmético: genera dos hijos como combinaciones lineales de los padres.
    
    child1 = α·parent1 + (1-α)·parent2
    child2 = (1-α)·parent1 + α·parent2
    
    donde α ∈ [0,1] es aleatorio en cada operación.
    
    Los hijos siempre caen ENTRE los padres, lo que es apropiado para
    representaciones de punto flotante en intervalos acotados.
    
    Limitación: no puede crear puntos fuera del rango [min(p1,p2), max(p1,p2)],
    por lo que la exploración depende principalmente de la mutación.
    """
    alpha = random.random()
    child1 = alpha * parent1 + (1 - alpha) * parent2
    child2 = (1 - alpha) * parent1 + alpha * parent2
    return child1, child2


# ─────────────────────────────────────────────────────────────────────────────
# 3.7 Mutación gaussiana
# ─────────────────────────────────────────────────────────────────────────────
def mutate(individual, mutation_rate, mutation_strength, lower_bound, upper_bound):
    """
    Introduce variación aleatoria en el individuo con probabilidad mutation_rate.
    
    El perturbación sigue una distribución N(0, mutation_strength):
    - Valores centrados en 0: no hay sesgo en dirección
    - mutation_strength controla cuán grandes pueden ser los saltos
    
    La mutación es el mecanismo principal de EXPLORACIÓN en este AG:
    permite escapar de mínimos locales y mantener diversidad genética.
    
    El clipping (max/min) garantiza que el individuo no salga del dominio.
    """
    if random.random() < mutation_rate:
        individual += random.gauss(0, mutation_strength)
        individual = max(lower_bound, min(upper_bound, individual))
    return individual


print("Funciones del AG definidas correctamente.")

---
## Sección 4 — Motor del AG: función que ejecuta una corrida completa

### ¿Por qué encapsular en una función?
El enunciado pide **repetir el experimento con 3 valores diferentes de SIGMA_SHARE**. En lugar de copiar el ciclo principal 3 veces, lo envolvemos en una función que recibe `sigma_share` como parámetro. Esto sigue el principio DRY (*Don't Repeat Yourself*) y facilita el análisis comparativo.

In [ ]:
def run_ag_fitness_sharing(
    objective_fn,
    sigma_share,
    lower_bound=0,
    upper_bound=10,
    population_size=100,
    num_generations=150,
    mutation_rate=0.1,
    mutation_strength=0.4,
    tournament_size=3,
    crossover_rate=0.8,
    alpha_sharing=1.0,
    seed=None
):
    """
    Ejecuta un AG completo con Fitness Sharing.
    
    Retorna:
    --------
    dict con:
      - 'population'      : población final (lista de valores x)
      - 'final_fitness'   : fitness original de la población final
      - 'best_per_gen'    : mejor fitness por generación (curva de convergencia)
      - 'avg_per_gen'     : fitness promedio por generación
      - 'sigma_share'     : el sigma usado (para trazabilidad)
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
    
    # Inicializar población
    population = initialize_population(population_size, lower_bound, upper_bound)
    
    best_per_gen = []
    avg_per_gen  = []
    
    # ─── Ciclo generacional ───────────────────────────────────────────────────
    for generation in range(num_generations):
        
        # Paso 1: Calcular fitness original (sin penalización de nicho)
        original_fitness = calculate_fitness(population, objective_fn)
        
        # Paso 2: Aplicar Fitness Sharing → fitness penalizado por densidad
        shared_fitness = apply_fitness_sharing(
            population, original_fitness, sigma_share, alpha_sharing
        )
        
        # Guardar estadísticas con fitness ORIGINAL (para ver el progreso real)
        best_per_gen.append(np.max(original_fitness))
        avg_per_gen.append(np.mean(original_fitness))
        
        # Paso 3: Crear nueva generación mediante selección + crossover + mutación
        new_population = []
        while len(new_population) < population_size:
            # Selección basada en fitness COMPARTIDO (clave del niching)
            parent1 = tournament_selection(population, shared_fitness, tournament_size)
            parent2 = tournament_selection(population, shared_fitness, tournament_size)
            
            if random.random() < crossover_rate:
                child1, child2 = crossover(parent1, parent2)
            else:
                child1, child2 = parent1, parent2
            
            child1 = mutate(child1, mutation_rate, mutation_strength, lower_bound, upper_bound)
            child2 = mutate(child2, mutation_rate, mutation_strength, lower_bound, upper_bound)
            
            new_population.append(child1)
            if len(new_population) < population_size:
                new_population.append(child2)
        
        population = new_population
    # ─── Fin ciclo generacional ───────────────────────────────────────────────
    
    final_fitness = calculate_fitness(population, objective_fn)
    
    return {
        'population'   : population,
        'final_fitness': final_fitness,
        'best_per_gen' : best_per_gen,
        'avg_per_gen'  : avg_per_gen,
        'sigma_share'  : sigma_share
    }


print("Función run_ag_fitness_sharing() definida.")

---
## Sección 5 — Función auxiliar de visualización

Genera el gráfico comparativo de función objetivo + distribución de la población final para un resultado dado. Se reutilizará para los tres experimentos.

In [ ]:
def extraer_optimos(population, final_fitness, sigma_share, min_fitness=0.1, max_optimos=6):
    """
    Heurística post-procesamiento para identificar los picos encontrados.
    
    Algoritmo:
    - Ordena individuos de mayor a menor fitness.
    - Itera: si un individuo está suficientemente lejos de todos los óptimos
      ya registrados (distancia > sigma_share), lo agrega como nuevo óptimo.
    
    Nota: esta es una aproximación — el verdadero análisis está en los gráficos.
    """
    sorted_idx = np.argsort(final_fitness)[::-1]
    optima_x, optima_y = [], []
    min_dist = sigma_share  # usar sigma_share como separación mínima entre óptimos
    
    for i in sorted_idx:
        cx, cy = population[i], final_fitness[i]
        if cy < min_fitness:
            continue
        es_nuevo = all(phenotypic_distance(cx, ox) >= min_dist for ox in optima_x)
        if es_nuevo:
            optima_x.append(cx)
            optima_y.append(cy)
        if len(optima_x) >= max_optimos:
            break
    
    return optima_x, optima_y


def plot_resultado(resultado, objective_fn, titulo_extra='', lower=0, upper=10):
    """
    Genera los dos gráficos estándar para un resultado del AG:
      1. Función objetivo + distribución de la población final
      2. Evolución del fitness a lo largo de las generaciones
    """
    sigma = resultado['sigma_share']
    population = resultado['population']
    final_fitness = resultado['final_fitness']
    
    optima_x, optima_y = extraer_optimos(population, final_fitness, sigma)
    
    x_plot = np.linspace(lower, upper, 1000)
    y_plot = objective_fn(x_plot)
    
    fig, axs = plt.subplots(2, 1, figsize=(13, 10))
    fig.suptitle(f"SIGMA_SHARE = {sigma}  |  {titulo_extra}", fontsize=14, fontweight='bold')
    
    # ── Gráfico 1: Función + Población ────────────────────────────────────────
    axs[0].plot(x_plot, y_plot, 'b-', linewidth=2, label='Función Objetivo', zorder=1)
    axs[0].scatter(
        population, final_fitness,
        color='red', s=25, alpha=0.6, label='Población Final', zorder=2
    )
    axs[0].scatter(
        optima_x, optima_y,
        color='lime', s=150, marker='*', edgecolors='black', linewidths=0.8,
        label=f'Óptimos detectados ({len(optima_x)})', zorder=3
    )
    # Marcar picos reales para referencia
    for px in PICOS_REALES['x']:
        axs[0].axvline(px, color='gray', linestyle=':', alpha=0.4)
    
    axs[0].set_xlabel('x')
    axs[0].set_ylabel('f(x)')
    axs[0].set_title('Función Objetivo y Distribución de la Población Final')
    axs[0].legend()
    axs[0].grid(True)
    
    # ── Gráfico 2: Evolución del fitness ──────────────────────────────────────
    gens = range(1, len(resultado['best_per_gen']) + 1)
    axs[1].plot(gens, resultado['best_per_gen'], color='green',  label='Mejor Fitness (original)')
    axs[1].plot(gens, resultado['avg_per_gen'],  color='orange', label='Fitness Promedio (original)')
    axs[1].set_xlabel('Generación')
    axs[1].set_ylabel('Fitness')
    axs[1].set_title('Evolución del Fitness por Generación')
    axs[1].legend()
    axs[1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nÓptimos detectados para SIGMA_SHARE = {sigma}:")
    for ox, oy in sorted(zip(optima_x, optima_y), key=lambda t: t[0]):
        print(f"  x = {ox:.4f},  f(x) = {oy:.4f}")
    print(f"  → Total de picos identificados: {len(optima_x)} de 5 picos reales")


print("Funciones de visualización definidas.")

---
## Sección 6 — Experimentos con SIGMA_SHARE

### Diseño experimental

La distancia entre los picos cercanos P2 (x=4.0) y P3 (x=4.7) es **d ≈ 0.7**.

Se prueban tres valores de `SIGMA_SHARE`:

| Experimento | SIGMA_SHARE | Relación con d=0.7 | Hipótesis |
|-------------|------------|--------------------|-----------|
| A | **1.5** | >> d (más del doble) | P2 y P3 se fusionarán en un solo nicho. Otros picos muy separados podrían también fusionarse. |
| B | **0.60** | ≈ d (ligeramente menor) | P2 y P3 podrían distinguirse. Valor de referencia del enunciado. |
| C | **0.25** | << d (menos de la mitad) | P2 y P3 se distinguen claramente, pero posiblemente el AG fragmenta demasiado, creando nichos artificiales. |

### Experimento A — SIGMA_SHARE = 1.5 (radio grande)

In [ ]:
resultado_A = run_ag_fitness_sharing(
    objective_fn=objective_function_nueva,
    sigma_share=1.5,
    num_generations=150,
    seed=SEED
)

plot_resultado(
    resultado_A,
    objective_function_nueva,
    titulo_extra='Radio GRANDE — mayor que la distancia entre picos cercanos'
)

### Análisis — Experimento A (SIGMA_SHARE = 1.5)

**¿Qué se observa?**

> _[Completa esta celda después de ejecutar el experimento. Guía de qué observar:]_
>
> - ¿En cuántos picos se concentra la población?
> - ¿Los picos P2 (x=4.0) y P3 (x=4.7) son detectados como uno solo o por separado?
> - ¿Hay picos reales que el algoritmo no identifica?

**¿Por qué ocurre esto?**

Con `SIGMA_SHARE = 1.5` (muy mayor que d=0.7), el radio del nicho es tan grande que P2 y P3 quedan dentro del mismo nicho mutuo. Esto significa que un individuo en P2 "ve" a un individuo en P3 como si fuera su vecino cercano, y vice versa. Ambos grupos se penalizan mutuamente, pero al estar en el mismo nicho, el AG no los distingue como óptimos separados. La presión hacia P2 y P3 se concentra en el punto de mayor altura entre ellos, absorbiendo a la subpoblación del más débil.

Adicionalmente, un radio grande puede hacer que picos más distantes (como P1 en x=1.5 y P4 en x=7.5) también interaccionen, dependiendo de la distribución de la población.

### Experimento B — SIGMA_SHARE = 0.60 (radio intermedio / ligeramente menor que d)

In [ ]:
resultado_B = run_ag_fitness_sharing(
    objective_fn=objective_function_nueva,
    sigma_share=0.60,
    num_generations=150,
    seed=SEED
)

plot_resultado(
    resultado_B,
    objective_function_nueva,
    titulo_extra='Radio INTERMEDIO — aproximadamente igual a la distancia entre picos cercanos'
)

### Análisis — Experimento B (SIGMA_SHARE = 0.60)

**¿Qué se observa?**

> _[Completa esta celda después de ejecutar el experimento]_
>
> - ¿Logra el algoritmo distinguir P2 y P3?
> - ¿Qué pasa con los picos más distantes (P1, P4, P5)?
> - ¿Cuántos picos totales identifica?

**¿Por qué ocurre esto?**

Con `SIGMA_SHARE = 0.60` (ligeramente menor que d=0.7), la distancia entre P2 y P3 queda justo fuera del radio de compartición. Esto significa que un individuo en P2 **no penaliza** directamente a uno en P3 (y viceversa), permitiendo que ambos picos sostengan sus propias subpoblaciones.

Este valor es el que mejor balancea los dos efectos: mantiene separados los picos cercanos y permite cobertura de los picos lejanos. Sin embargo, en la práctica este equilibrio es frágil, y resultados pueden variar entre corridas si no se usa semilla fija.

### Experimento C — SIGMA_SHARE = 0.25 (radio pequeño)

In [ ]:
resultado_C = run_ag_fitness_sharing(
    objective_fn=objective_function_nueva,
    sigma_share=0.25,
    num_generations=150,
    seed=SEED
)

plot_resultado(
    resultado_C,
    objective_function_nueva,
    titulo_extra='Radio PEQUEÑO — significativamente menor que la distancia entre picos cercanos'
)

### Análisis — Experimento C (SIGMA_SHARE = 0.25)

**¿Qué se observa?**

> _[Completa esta celda después de ejecutar el experimento]_
>
> - ¿Están todos los picos representados o algunos quedan sin cobertura?
> - ¿Hay fragmentación dentro de un mismo pico (múltiples clústeres en un pico real)?
> - ¿Cómo es la dispersión de la población comparada con los experimentos A y B?

**¿Por qué ocurre esto?**

Con `SIGMA_SHARE = 0.25`, el radio de cada individuo es muy pequeño. Dos individuos en el mismo pico pero separados por más de 0.25 unidades **no se penalizan entre sí**, lo que puede generar múltiples "sub-nichos" dentro de un mismo pico real. El fitness sharing pierde capacidad de concentrar a los individuos en los óptimos exactos, y la población puede quedar más dispersa.

Paradójicamente, con radio muy pequeño, el AG se comporta casi como un AG sin fitness sharing: los individuos no se "comunican" tanto entre sí, y la presión selectiva normal domina. Esto puede llevar a convergencia prematura en los picos más altos.

---
## Sección 7 — Comparación visual de los tres experimentos

In [ ]:
x_plot = np.linspace(0, 10, 1000)
y_plot = objective_function_nueva(x_plot)

fig, axs = plt.subplots(3, 1, figsize=(13, 15))
fig.suptitle("Comparación de la distribución final según SIGMA_SHARE", fontsize=14, fontweight='bold')

configs = [
    (resultado_A, 'A — SIGMA_SHARE = 1.5 (grande)', 'tomato'),
    (resultado_B, 'B — SIGMA_SHARE = 0.60 (intermedio)', 'royalblue'),
    (resultado_C, 'C — SIGMA_SHARE = 0.25 (pequeño)', 'mediumseagreen'),
]

for ax, (resultado, titulo, color) in zip(axs, configs):
    sigma = resultado['sigma_share']
    pop   = resultado['population']
    fit   = resultado['final_fitness']
    opt_x, opt_y = extraer_optimos(pop, fit, sigma)
    
    ax.plot(x_plot, y_plot, 'b-', linewidth=2, alpha=0.6, label='f(x)', zorder=1)
    ax.scatter(pop, fit, color=color, s=20, alpha=0.5, label='Población Final', zorder=2)
    ax.scatter(opt_x, opt_y, color='gold', s=200, marker='*', edgecolors='black',
               linewidths=0.8, label=f'Óptimos ({len(opt_x)})', zorder=3)
    for px in PICOS_REALES['x']:
        ax.axvline(px, color='gray', linestyle=':', alpha=0.35)
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True)

plt.tight_layout()
plt.show()

---
## Sección 8 — Discusión: sensibilidad al parámetro SIGMA_SHARE

### ¿Cuál es el valor "adecuado" de SIGMA_SHARE para esta función?

> _[Completa esta celda con tu análisis personal, basado en los tres gráficos]_

**Resumen esperado de resultados:**

| SIGMA_SHARE | Picos detectados | Observación principal |
|-------------|-----------------|----------------------|
| 1.5 (grande)    | ≤ 4 | P2 y P3 se fusionan en un nicho. Posible pérdida de P5 (más débil). |
| 0.60 (intermedio) | ≈ 5 | Mejor separación de todos los picos, incluyendo P2 y P3. |
| 0.25 (pequeño)   | variable | Dispersión excesiva; puede no concentrarse bien en los óptimos. |

**¿Por qué SIGMA_SHARE = 0.60 sería el más adecuado?**

La regla general es que `SIGMA_SHARE` debe ser **del orden de la mitad de la distancia mínima entre óptimos que se quiere distinguir**. En nuestra función, la distancia mínima entre picos es ~0.7 (P2–P3). Un valor de 0.60 está justo por debajo de esa distancia, lo que permite que los picos no se "contaminen" mutuamente, pero al mismo tiempo es lo suficientemente grande como para que varios individuos en el mismo pico sí se penalicen entre sí, promoviendo una distribución uniforme dentro de cada nicho.

**Sensibilidad del algoritmo:**

El AG con Fitness Sharing es **muy sensible** a `SIGMA_SHARE`. Valores incorrectos tienen consecuencias opuestas:
- Demasiado grande → **fusión de nichos** (pérdida de resolución en el espacio de soluciones)
- Demasiado pequeño → **fragmentación de nichos** (múltiples representantes del mismo pico real, pérdida de picos de menor altura)

En la práctica, cuando no se conoce a priori la separación entre óptimos (lo más frecuente en problemas reales), se recomienda hacer un **barrido de valores** o usar métodos adaptativos de SIGMA_SHARE.

---
## Sección 9 — Planteamiento en entorno laboral

### Problema: Optimización de pipelines de evaluación ex-post con AG

> _[Adapta esta sección a tu propio contexto — la siguiente es una propuesta basada en tu trabajo]_

**Contexto:**  
En el pipeline de evaluación ex-post del CAF (`caf_eval_ex_post_ai`), cada proyecto de inversión es evaluado mediante un conjunto de preguntas que el modelo GPT-4o responde según criterios cualitativos. El pipeline produce un puntaje global por proyecto. Sin embargo, **la selección del prompt** (instrucción al modelo) impacta significativamente la calidad de las respuestas.

**Problema abordable con AG:**  
Encontrar múltiples configuraciones de prompt (combinaciones de estructura, tono, nivel de detalle solicitado y criterios de evaluación explícitos) que produzcan evaluaciones de alta calidad según una métrica objetiva (e.g., correlación con evaluaciones humanas), sin converger a una sola configuración "óptima".

---

**Función objetivo:**

$$f(\text{prompt}) = w_1 \cdot \text{Correlación}(\hat{y}, y_{\text{humano}}) + w_2 \cdot \text{Coherencia}(\hat{y}) - w_3 \cdot \text{Longitud\_exceso}(\hat{y})$$

donde:
- $\hat{y}$ = respuesta generada por el LLM dado el prompt
- $y_{\text{humano}}$ = evaluación de referencia humana
- $\text{Coherencia}$ = métrica de consistencia lógica entre secciones
- $\text{Longitud\_exceso}$ = penalización si la respuesta es demasiado larga o demasiado corta

---

**Estructura del cromosoma:**

Cada individuo (cromosoma) representa una configuración de prompt:

| Gen | Nombre | Tipo | Rango / Valores | Descripción |
|-----|--------|------|-----------------|-------------|
| g1 | `estructura` | Entero | {0, 1, 2} | 0=libre, 1=por dimensiones, 2=por criterios CAF |
| g2 | `nivel_detalle` | Real | [0.0, 1.0] | 0=resumen, 1=detalle exhaustivo |
| g3 | `incluir_ejemplos` | Binario | {0, 1} | 1=incluir ejemplos en el prompt |
| g4 | `tono` | Entero | {0, 1, 2} | 0=técnico, 1=ejecutivo, 2=académico |
| g5 | `temperatura_llm` | Real | [0.1, 1.0] | Temperatura del modelo de lenguaje |

**Genotipo:** Vector mixto → `[g1, g2, g3, g4, g5]` = e.g. `[1, 0.7, 1, 0, 0.3]`

**Fenotipo:** El prompt completo resultante de decodificar ese cromosoma y la respuesta generada por el LLM.

**¿Por qué Fitness Sharing?**  
Interesa encontrar **múltiples configuraciones válidas** (no solo la "mejor"), porque diferentes tipos de proyecto (PPI, infraestructura, social) pueden requerir prompts distintos. El AG con niching permite identificar simultáneamente las mejores configuraciones para cada tipo, sin que una domine y elimine a las demás.

---
## Conclusiones

1. **Fitness Sharing** es una técnica efectiva de niching que permite a los Algoritmos Genéticos identificar múltiples óptimos en funciones multimodales, redistribuyendo la presión selectiva de manera proporcional a la densidad de población en cada región del espacio de búsqueda.

2. **El parámetro SIGMA_SHARE es crítico** y su selección adecuada depende directamente de la separación entre los óptimos que se desea distinguir. Una regla práctica es usar un valor igual o ligeramente inferior a la mitad de la distancia mínima entre picos de interés.

3. Los **efectos de un valor inadecuado** son claros:
   - **Grande:** fusión de nichos → pérdida de resolución multimodal
   - **Pequeño:** fragmentación → comportamiento cercano a un AG estándar, con riesgo de convergencia prematura

4. En aplicaciones reales con espacio de búsqueda desconocido, se recomienda **exploración previa** del paisaje (landscape analysis) o técnicas de SIGMA_SHARE adaptativo para no depender de una calibración manual.